# LookBackLens Baseline — Attention-based Hallucination Detection

An attention-based token-level lookup baseline that extracts lookback ratios from the decoder's self-attention patterns. Using a pre-trained autoregressive model (`Qwen/Qwen2.5-3B-Instruct`), the system calculates the ratio of attention directed toward the context block versus the newly generated answer tokens across all model layers. 

Feature aggregation is performed over token indices using a sliding-window mechanism. A Logistic Regression classifier is then trained on these window features using the `combined_train.jsonl` split. This classifier maps attention patterns to binary labels, which are subsequently decoded back into character-level spans to detect hallucinations on the test splits.

In [1]:
!pip install --upgrade pip

!pip install torch --index-url https://download.pytorch.org/whl/cu121

!pip install transformers accelerate scikit-learn joblib pandas numpy tqdm ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.2 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 22.3.1
    Uninstalling pip-22.3.1:
      Successfully uninstalled pip-22.3.1
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 53.1 MB/s  0:00:13:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 60.4 MB/s  0:00:00m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 64.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 51.6 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 73.6 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 77.7 MB/s  0:00:05:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 83.3 MB/s  0:00:01:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 100.8 MB/s  0:00:00eta 0:00:

In [2]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass
from sklearn.linear_model import LogisticRegression
import joblib

os.makedirs("/app/results", exist_ok=True)
os.makedirs("/app/checkpoints", exist_ok=True)
DATA_DIR = Path("/app/data")

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct" 
MAX_LENGTH = 4096
WINDOW, STRIDE = 8, 8
THRESHOLD = 0.5
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

TEST_FILES = {
    "Combined": DATA_DIR / "combined_test.jsonl",
    "Type 1 + clean": DATA_DIR / "type1_test_balanced.jsonl",
    "Type 2 + clean": DATA_DIR / "type2_test_balanced.jsonl",
    "Type 3 + clean": DATA_DIR / "type3_test_balanced.jsonl",
}

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def _char_set(spans):
    chars = set()
    for s in spans: chars.update(range(int(s["start"]), int(s["end"])))
    return chars

def span_metrics(samples, pred_spans):
    micro_overlap = micro_pred = micro_gold = 0
    macro_f1 = []
    for sample, preds in zip(samples, pred_spans):
        gold = _char_set(sample.get("hallucination_labels", []))
        pred = _char_set(preds)
        overlap = len(gold & pred)
        micro_overlap += overlap; micro_pred += len(pred); micro_gold += len(gold)
        if not pred and not gold:
            macro_f1.append(1.0); continue
        p = overlap / len(pred) if pred else 0.0
        r = overlap / len(gold) if gold else 0.0
        macro_f1.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    p_mi = micro_overlap / micro_pred if micro_pred else 0.0
    r_mi = micro_overlap / micro_gold if micro_gold else 0.0
    f_mi = 2 * p_mi * r_mi / (p_mi + r_mi) if (p_mi + r_mi) > 0 else 0.0
    return {"P": p_mi, "R": r_mi, "F1": f_mi, "macro_F1": sum(macro_f1)/len(macro_f1) if macro_f1 else 0.0}

def example_metrics(samples, pred_spans):
    tp = fp = fn = 0
    for sample, preds in zip(samples, pred_spans):
        gold = 1 if sample.get("hallucination_labels") else 0
        pred = 1 if preds else 0
        if gold and pred: tp += 1
        elif gold: fn += 1
        elif pred: fp += 1
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    return {"P": p, "R": r, "F1": 2 * p * r / (p + r) if (p + r) > 0 else 0.0}

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, attn_implementation="eager", device_map="cuda:0"
)
model.eval()

@dataclass
class ExtractionOut:
    ratios: np.ndarray
    answer_token_offsets: np.ndarray

@torch.no_grad()
def extract_lookback_ratios(model, tokenizer, sample, device="cuda:0", max_length=MAX_LENGTH):
    context_block = f"{sample.get('context', '')}\n\nQuestion: {sample.get('query', '')}\n\nAnswer: "
    enc_ctx = tokenizer(context_block, add_special_tokens=True, return_tensors="pt")
    enc_ans = tokenizer(sample.get("output", ""), add_special_tokens=False, return_tensors="pt", return_offsets_mapping=True)
    
    input_ids = torch.cat([enc_ctx["input_ids"], enc_ans["input_ids"]], dim=1)
    if input_ids.size(1) > max_length:
        input_ids = torch.cat([enc_ctx["input_ids"][:, :max(0, enc_ctx["input_ids"].size(1) - (input_ids.size(1) - max_length))], enc_ans["input_ids"]], dim=1)
        
    attention_mask = torch.ones_like(input_ids)
    context_end = input_ids.size(1) - enc_ans["input_ids"].size(1)
    num_ans = enc_ans["input_ids"].size(1)

    out = model(input_ids=input_ids.to(device), attention_mask=attention_mask.to(device), output_attentions=True, use_cache=False)
    
    tril = torch.tril(torch.ones(num_ans, num_ans, device=device, dtype=torch.float32), diagonal=-1)
    counts = torch.arange(num_ans, device=device, dtype=torch.float32).clamp(min=1)
    ratios = torch.empty(len(out.attentions), out.attentions[0].size(1), num_ans, device=device, dtype=torch.float32)
    
    for l, attn in enumerate(out.attentions):
        ans_rows = attn[0].to(torch.float32)[:, context_end:context_end + num_ans, :]
        att_ctx = ans_rows[:, :, :context_end].mean(dim=-1)
        att_new = (ans_rows[:, :, context_end:context_end + num_ans] * tril).sum(dim=-1) / counts
        att_new[:, 0] = 0
        ratios[l] = att_ctx / (att_ctx + att_new + 1e-6)

    return ExtractionOut(ratios.cpu().numpy(), enc_ans["offset_mapping"][0].numpy())

def sliding_windows(ratios, window, stride):
    n = ratios.shape[-1]
    if n == 0: return np.zeros((0, ratios.shape[0]*ratios.shape[1]), dtype=np.float32), []
    if n < window: window = n
    feats, idx_lists = [], []
    for s in range(0, n - window + 1, stride):
        idxs = list(range(s, s + window))
        feats.append(ratios[:, :, idxs].mean(axis=-1).reshape(-1).astype(np.float32))
        idx_lists.append(idxs)
    return np.stack(feats) if feats else np.array([]), idx_lists

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
train_samples = read_jsonl(DATA_DIR / "combined_train.jsonl")
train_X, train_y = [], []

for s in tqdm(train_samples, desc="Обучение LogisticRegression"):
    out = extract_lookback_ratios(model, tokenizer, s)
    feats, idx_lists = sliding_windows(out.ratios, WINDOW, STRIDE)
    if len(feats) == 0: continue
    
    gold_idxs = set()
    for span in s.get("hallucination_labels", []):
        gold_idxs.update([i for i, (a, b) in enumerate(out.answer_token_offsets) if a < int(span["end"]) and b > int(span["start"]) and a != b])
        
    for feat, idxs in zip(feats, idx_lists):
        train_X.append(feat); train_y.append(1 if any(i in gold_idxs for i in idxs) else 0)

train_X, train_y = np.stack(train_X, axis=0), np.array(train_y)
clf = LogisticRegression(max_iter=2000, solver="lbfgs", random_state=SEED)
clf.fit(train_X, train_y)
joblib.dump(clf, "/app/checkpoints/lookback_clf.pkl")
print(f"Модель обучена (Acc: {clf.score(train_X, train_y):.3f})")

Обучение LogisticRegression:   0%|          | 0/9893 [00:00<?, ?it/s]

Модель обучена (Acc: 0.963)


In [5]:
def windows_to_spans(offsets, idx_lists, scores, threshold):
    spans, cs, ce = [], None, None
    for idxs, sc in zip(idx_lists, scores):
        if sc > threshold:
            c_s, c_e = int(offsets[idxs[0], 0]), int(offsets[idxs[-1], 1])
            if cs is None: cs, ce = c_s, c_e
            elif c_s <= ce: ce = max(ce, c_e)
            else: spans.append({"start": cs, "end": ce}); cs, ce = c_s, c_e
        else:
            if cs is not None: spans.append({"start": cs, "end": ce}); cs = ce = None
    if cs is not None: spans.append({"start": cs, "end": ce})
    return spans

results = []
for name, path in TEST_FILES.items():
    if not path.exists(): continue
    samples = read_jsonl(path)
    preds = []
    for s in tqdm(samples, desc=name):
        out = extract_lookback_ratios(model, tokenizer, s)
        feats, idx_lists = sliding_windows(out.ratios, WINDOW, STRIDE)
        if len(feats) == 0: preds.append([]); continue
        scores = clf.predict_proba(feats)[:, 1]
        preds.append(windows_to_spans(out.answer_token_offsets, idx_lists, scores, THRESHOLD))
        
    sm = span_metrics(samples, preds)
    em = example_metrics(samples, preds)
    results.append({
        "Split": name, "N": len(samples),
        "Span P": round(sm["P"], 3), "Span R": round(sm["R"], 3), "Span F1": round(sm["F1"], 3), "Span macro F1": round(sm["macro_F1"], 3),
        "Ex P": round(em["P"], 3), "Ex R": round(em["R"], 3), "Ex F1": round(em["F1"], 3)
    })

with open("/app/results/lookback_baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

display(pd.DataFrame(results))

Combined:   0%|          | 0/599 [00:00<?, ?it/s]

Type 1 + clean:   0%|          | 0/300 [00:00<?, ?it/s]

Type 2 + clean:   0%|          | 0/300 [00:00<?, ?it/s]

Type 3 + clean:   0%|          | 0/299 [00:00<?, ?it/s]

,Split,N,Span P,Span R,Span F1,Span macro F1,Ex P,Ex R,Ex F1
0,Combined,599,0.746,0.797,0.771,0.679,0.938,0.875,0.906
1,Type 1 + clean,300,0.206,0.502,0.293,0.545,0.789,0.647,0.711
2,Type 2 + clean,300,0.791,0.821,0.805,0.828,0.851,0.987,0.914
3,Type 3 + clean,299,0.729,0.808,0.766,0.812,0.851,0.993,0.916
